In [0]:
%run ../../02_common_utils/operations

In [0]:
catalog           = "charles_schwab_retailbrokerage_dev_team_lemma"
staging_prospect  = f"{catalog}.staging.prospect_current"
gold_dim_customer = f"{catalog}.gold.dim_customer"
gold_dim_prospect = f"{catalog}.gold.dim_prospect"

dbutils.widgets.text("batch_id", "1", "Batch ID")
current_batch = dbutils.widgets.get("batch_id")

try:
    run_info_row = spark.sql(f"SELECT _run_id, _batch FROM {staging_prospect} LIMIT 1").first()
    carried_run_id = run_info_row[0] if run_info_row else "unknown"
    carried_batch = run_info_row[1] if run_info_row else "unknown"
except Exception:
    carried_run_id = "unknown"
    carried_batch = "unknown"


In [0]:
log_pipeline_message(spark, carried_run_id, 'INFO', 'gold_customer_prospect', 'Starting processing for gold dim_prospect')
start_pipeline_run(spark, carried_run_id, carried_batch)
log_domain_run_status(spark, carried_run_id, carried_batch, 'PROSPECT', 'RUNNING')

In [0]:
# ─── BUILD AND OVERWRITE ─────────────────────────────────────────────────
spark.sql(f"""
    WITH ProspectBase AS (
        SELECT 
            p.agency_id as agencyid,
            TRY_CAST(DATE_FORMAT(CURRENT_DATE(), 'yyyyMMdd') AS BIGINT) as sk_recorddateid,
            TRY_CAST(DATE_FORMAT(CURRENT_DATE(), 'yyyyMMdd') AS BIGINT) as sk_updatedateid,
            p.first_batchid as batchid,
            
            -- Core Strings (No NULLs in source)
            p.last_name as lastname, 
            p.first_name as firstname, 
            p.city, 
            p.state, 
            
            -- 1. Categorical Flags (Handled as 'U')
            COALESCE(UPPER(p.gender), 'U') as gender,
            COALESCE(UPPER(p.own_or_rent), 'U') as ownorrentflag,
            
            -- 2. Descriptive Text Masking
            COALESCE(UPPER(p.middle_initial), 'N/A') as middleinitial, 
            COALESCE(p.address_line1, 'Unknown') as addressline1, 
            COALESCE(p.address_line2, 'N/A') as addressline2, 
            COALESCE(p.postal_code, 'Unknown') as postalcode,
            COALESCE(p.country, 'Unknown') as country, 
            COALESCE(p.phone_full_number, 'Unknown') as phone,
            COALESCE(UPPER(p.marital_status), 'Unknown') as maritalstatus, 
            COALESCE(p.employer_name, 'Unknown') as employer,
            
            -- 3. Numerics & Financials (Safely Kept as NULL)
            TRY_CAST(p.annual_income AS DECIMAL(15,2)) as income,
            TRY_CAST(p.number_cars AS INT) as numbercars, 
            TRY_CAST(p.number_children AS INT) as numberchildren,
            TRY_CAST(p.age AS INT) as age,
            TRY_CAST(p.credit_rating AS INT) as creditrating, 
            TRY_CAST(p.number_credit_cards AS INT) as numbercreditcards,
            TRY_CAST(p.net_worth AS DECIMAL(15,2)) as networth,
            
            -- Marketing Nameplate (Empty strings convert to 'Unknown Profile')
            COALESCE(NULLIF(CONCAT_WS('+',
                CASE WHEN TRY_CAST(net_worth AS DECIMAL(15,2)) > 1000000 OR TRY_CAST(annual_income AS DECIMAL(15,2)) > 200000 THEN 'HighValue' ELSE NULL END,
                CASE WHEN TRY_CAST(number_children AS INT) > 3 THEN 'Expenses' ELSE NULL END,
                CASE WHEN TRY_CAST(age AS INT) BETWEEN 45 AND 67 THEN 'Boomer' ELSE NULL END,
                CASE WHEN TRY_CAST(age AS INT) > 67 THEN 'Elderly' ELSE NULL END,
                CASE WHEN TRY_CAST(age AS INT) < 25 THEN 'YoungAdult' ELSE NULL END,
                CASE WHEN TRY_CAST(net_worth AS DECIMAL(15,2)) BETWEEN 100000 AND 1000000 THEN 'Millennial' ELSE NULL END,
                CASE WHEN TRY_CAST(number_credit_cards AS INT) > 5 THEN 'Spender' ELSE NULL END,
                CASE WHEN own_or_rent = 'R' THEN 'Renter' ELSE NULL END
            ), ''), 'Unknown Profile') AS marketingnameplate,
            
            p._batch, 
            CURRENT_TIMESTAMP() as _load_ts, 
            p._run_id
        FROM {staging_prospect} p
    )
    SELECT 
        pb.*,
        -- FIX: Use EXISTS to prevent row duplication if multiple customers share the same name/address
        CASE WHEN EXISTS (
            SELECT 1 FROM {gold_dim_customer} c
            WHERE UPPER(pb.lastname) = UPPER(c.lastname) 
              AND UPPER(pb.firstname) = UPPER(c.firstname)
              AND UPPER(pb.addressline1) = UPPER(c.addressline1) 
              AND UPPER(COALESCE(pb.addressline2, '')) = UPPER(COALESCE(c.addressline2, ''))
              AND UPPER(pb.postalcode) = UPPER(c.postalcode)
              AND c.iscurrent = TRUE
        ) THEN TRUE ELSE FALSE END as iscustomer
    FROM ProspectBase pb
""").write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(gold_dim_prospect)

dim_prospect_count = spark.sql(f"SELECT COUNT(*) FROM {gold_dim_prospect}").first()[0]
print(f"gold.dim_prospect rows: {dim_prospect_count} (Expected: 49,940)")

In [0]:
# ─── LOGGING ─────────────────────────────────────────────────────────────
staging_count = spark.sql(f"SELECT COUNT(*) FROM {staging_prospect}").first()[0]
log_audit_event(spark, carried_run_id, current_batch, "gold", "dim_prospect", "OVERWRITE", dim_prospect_count)
log_pipeline_recon(spark, carried_run_id, current_batch, "CUSTOMER", "dim_prospect", "staging", "gold", staging_count, dim_prospect_count)


log_gold_recon(spark, carried_run_id, "gold.dim_prospect", expected_count=49940, actual_count=dim_prospect_count)

log_domain_run_status(spark, carried_run_id, carried_batch, 'PROSPECT', 'COMPLETED')
end_pipeline_run(spark, carried_run_id, 'SUCCESS')
log_pipeline_message(spark, carried_run_id, 'INFO', 'gold_customer_prospect', 'Successfully completed processing for gold dim_prospect.')